# Virelion-DCCP — Direct Runtime Bug Hunt

**Run the single code cell below.** It contains the setup and the complete runtime/integration test suite. There is no generated runner script, no virtual environment, no editable install, and no dependency on previous notebook cells.

Validation dependencies are installed only into `/content/DCCP_RUNTIME_DIRECT_DEPS`; Colab's global package environment is not upgraded.


In [ ]:
# DIRECT runtime bug-hunt: no generated runner, no venv, no editable install.
# Run this ONE cell from a fresh Colab runtime.

from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys
import tempfile
import traceback
import importlib
import re

# ---------------------------------------------------------------------
# 1. Fresh checkout and isolated dependency directory.
# ---------------------------------------------------------------------
ROOT = Path("/content/DCCP_RUNTIME_DIRECT")
DEPS = Path("/content/DCCP_RUNTIME_DIRECT_DEPS")

# Never delete the current working directory. Colab normally starts in /content.
# We explicitly run destructive setup commands from /content.
for path in (ROOT, DEPS):
    if path.exists():
        shutil.rmtree(path)

subprocess.run(
    [
        "git", "clone", "--branch", "main", "--depth", "1",
        "https://github.com/Virelion-Biotech/Virelion-DCCP.git",
        str(ROOT),
    ],
    cwd="/content",
    check=True,
)

DEPS.mkdir(parents=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check",
        "--no-warn-script-location",
        "--target", str(DEPS),
        "jsonschema",
        "pytest",
        "pytest-cov",
        "coverage",
        "hypothesis",
        "ruff==0.16.7",
    ],
    cwd="/content",
    check=True,
)

COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=ROOT,
    text=True,
).strip()

print("=" * 90)
print("VIRELION-DCCP DIRECT RUNTIME BUG-HUNT")
print("=" * 90)
print("Commit:", COMMIT)
print("Colab Python:", sys.version)
print("Repo:", ROOT)
print("Dependencies:", DEPS)

# ---------------------------------------------------------------------
# 2. Configure the notebook kernel and child processes.
# ---------------------------------------------------------------------
for path in (DEPS, ROOT / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

importlib.invalidate_caches()

TEST_ENV = os.environ.copy()
TEST_ENV["PYTHONPATH"] = os.pathsep.join(
    [str(DEPS), str(ROOT / "src")]
)
TEST_ENV["PYTHONNOUSERSITE"] = "1"

results = []

def check(name, fn):
    try:
        fn()
        results.append({"test": name, "passed": True, "detail": ""})
        print("PASS ::", name)
    except Exception as exc:
        detail = "".join(
            traceback.format_exception(type(exc), exc, exc.__traceback__)
        )
        results.append({
            "test": name,
            "passed": False,
            "detail": detail,
        })
        print("FAIL ::", name)
        print(detail)

def run_child(args, timeout=180):
    return subprocess.run(
        [sys.executable, *args],
        cwd=ROOT,
        env=TEST_ENV,
        text=True,
        capture_output=True,
        timeout=timeout,
    )

# ---------------------------------------------------------------------
# 3. Basic package identity.
# ---------------------------------------------------------------------
def test_package_identity():
    import dccp
    assert str(ROOT / "src") in str(dccp.__file__), dccp.__file__

check("package imports from checked-out source", test_package_identity)

# ---------------------------------------------------------------------
# 4. Import every module in a fresh child interpreter.
# ---------------------------------------------------------------------
def test_all_modules_import():
    modules = [
        p.stem
        for p in sorted((ROOT / "src" / "dccp").glob("*.py"))
        if p.name != "__init__.py"
    ]
    failures = []

    for module in modules:
        p = run_child(["-c", f"import dccp.{module}"])
        if p.returncode != 0:
            failures.append(
                f"dccp.{module}\n{p.stdout}\n{p.stderr}"
            )

    assert not failures, "\n\n".join(failures)

check("all DCCP modules import in fresh interpreters", test_all_modules_import)

# ---------------------------------------------------------------------
# 5. Load + audit every scenario.
# ---------------------------------------------------------------------
def test_all_scenarios():
    from dccp.scenario import load_scenario
    from dccp.audit import audit_scenario

    root = ROOT / "scenarios"
    files = sorted(root.rglob("*.json"))
    assert files, "No scenario JSON files found"

    failures = []
    for path in files:
        try:
            scenario = load_scenario(path)
            audit = audit_scenario(scenario)
            if not audit.passed:
                failures.append({
                    "path": str(path.relative_to(ROOT)),
                    "schema_errors": audit.schema_errors,
                    "policy_errors": audit.policy_errors,
                    "warnings": audit.policy_warnings,
                })
        except Exception as exc:
            failures.append({
                "path": str(path.relative_to(ROOT)),
                "exception": repr(exc),
            })

    assert not failures, json.dumps(failures, indent=2)
    print("  scenarios:", len(files))

check("every repository scenario loads and audits", test_all_scenarios)

# ---------------------------------------------------------------------
# 6. Registry / library / challenge / bundle integration.
# ---------------------------------------------------------------------
def test_infrastructure():
    from dccp.registry import build_registry, write_registry
    from dccp.library import (
        discover_scenarios,
        load_library,
        materialize_challenge_set,
        write_challenge_set,
    )
    from dccp.bundle import build_bundle

    root = ROOT / "scenarios"

    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td)
        files = discover_scenarios(root)
        assert files

        registry = build_registry(
            root,
            exclude_paths=[tmp / "registry.json"],
        )
        payload = write_registry(root, tmp / "registry.json")
        assert len(registry) == len(files)
        assert payload["n_entries"] == len(files)

        loaded_registry = json.loads(
            (tmp / "registry.json").read_text(encoding="utf-8")
        )
        assert loaded_registry["n_entries"] == len(files)

        library = load_library(root)
        assert len(library) == len(files)

        challenge = materialize_challenge_set(root)
        assert challenge["n_cases"] == len(library)
        assert isinstance(challenge["set_hash"], str)
        assert len(challenge["set_hash"]) == 64

        write_challenge_set(tmp / "challenge.json", root)
        written = json.loads(
            (tmp / "challenge.json").read_text(encoding="utf-8")
        )
        assert written["set_hash"] == challenge["set_hash"]

        bundle = build_bundle(
            tmp / "bundle",
            run_id="direct-colab-test",
            input_files=[files[0], files[0]],
            producer_version="0.3.0",
            base_dir=ROOT,
        )
        assert len(bundle["inputs"]) == 1
        assert not bundle["inputs"][0]["path"].startswith("/")
        assert (tmp / "bundle" / "manifest.json").exists()

        print("  scenarios:", len(files))

check(
    "registry/library/challenge-set/bundle integration",
    test_infrastructure,
)

# ---------------------------------------------------------------------
# 7. Defensive edge cases.
# ---------------------------------------------------------------------
def test_numeric_and_path_edges():
    from dccp.fingerprint import canonical_json
    from dccp.bundle import build_bundle

    for value in (float("nan"), float("inf"), float("-inf")):
        try:
            canonical_json({"x": value})
        except (ValueError, TypeError):
            pass
        else:
            raise AssertionError(
                f"canonical_json accepted non-finite value {value!r}"
            )

    with tempfile.TemporaryDirectory() as td:
        root = Path(td)
        base = root / "base"
        base.mkdir()
        outside = root / "outside.txt"
        outside.write_text("outside", encoding="utf-8")

        try:
            build_bundle(
                base / "bundle",
                run_id="outside-test",
                input_files=[outside],
                base_dir=base,
            )
        except (ValueError, FileNotFoundError, RuntimeError):
            pass
        else:
            raise AssertionError(
                "build_bundle accepted an input outside base_dir"
            )

check("numeric and path-safety edge cases", test_numeric_and_path_edges)

# ---------------------------------------------------------------------
# 8. Scenario API malformed-input behavior.
# ---------------------------------------------------------------------
def test_malformed_scenario_behavior():
    from dccp.scenario import validate_scenario

    assert validate_scenario(None)
    assert validate_scenario([])

    errors = validate_scenario({})
    assert errors

check("malformed scenario inputs are rejected", test_malformed_scenario_behavior)

# ---------------------------------------------------------------------
# 9. Exact CI Ruff check.
# ---------------------------------------------------------------------
def test_ruff():
    p = subprocess.run(
        [sys.executable, "-m", "ruff", "check", "src", "tests"],
        cwd=ROOT,
        env=TEST_ENV,
        text=True,
        capture_output=True,
    )
    assert p.returncode == 0, (p.stdout + "\n" + p.stderr)[-12000:]

check("Ruff 0.16.7 exact CI check", test_ruff)

# ---------------------------------------------------------------------
# 10. Exact pytest suite.
# ---------------------------------------------------------------------
def test_pytest():
    p = subprocess.run(
        [sys.executable, "-m", "pytest", "-q"],
        cwd=ROOT,
        env=TEST_ENV,
        text=True,
        capture_output=True,
        timeout=300,
    )
    assert p.returncode == 0, (p.stdout + "\n" + p.stderr)[-16000:]
    print(p.stdout)

check("pytest -q", test_pytest)

# ---------------------------------------------------------------------
# 11. CLI commands used by the repository CI.
# ---------------------------------------------------------------------
CLI_COMMANDS = [
    [
        "-m", "dccp.cli", "--version",
    ],
    [
        "-m", "dccp.cli", "--help",
    ],
    [
        "-m", "dccp.cli",
        "validate",
        "scenarios/examples/SCENARIO-001.ordinary-mi.json",
    ],
    [
        "-m", "dccp.cli",
        "audit-all",
        "scenarios/examples",
    ],
    [
        "-m", "dccp.cli",
        "registry",
        "--root", "scenarios/examples",
        "--output", "/tmp/dccp-direct-registry.json",
    ],
    [
        "-m", "dccp.cli",
        "release-gate",
        "/tmp/dccp-direct-registry.json",
        "--base", ".",
    ],
    [
        "-m", "dccp.cli",
        "materialize",
        "--root", "scenarios/examples",
        "--output", "/tmp/dccp-direct-challenge.json",
    ],
    [
        "-m", "dccp.cli",
        "map-scores",
        "--scores", "inflammatory=0.8,contractile_functional=0.4",
        "--draft-id", "SCENARIO-902",
        "-o", "/tmp/dccp-direct-draft.json",
    ],
    [
        "-m", "dccp.cli",
        "validate",
        "/tmp/dccp-direct-draft.json",
    ],
    [
        "-m", "dccp.cli",
        "bundle",
        "--output-dir", "/tmp/dccp-direct-bundle",
        "--run-id", "ci",
        "--input-files",
        "scenarios/examples/SCENARIO-001.ordinary-mi.json",
    ],
]

def test_cli():
    failures = []

    for command in CLI_COMMANDS:
        p = run_child(command, timeout=180)
        if p.returncode != 0:
            failures.append({
                "command": command,
                "returncode": p.returncode,
                "stdout": p.stdout,
                "stderr": p.stderr,
            })

    assert not failures, json.dumps(failures, indent=2)

check("all CI CLI smoke commands", test_cli)

# ---------------------------------------------------------------------
# 12. Repeated deterministic library loading.
# ---------------------------------------------------------------------
def test_repeated_determinism():
    from dccp.library import load_library
    from dccp.provenance import canonical_hash

    root = ROOT / "scenarios"
    observed = []

    for _ in range(50):
        library = load_library(root)
        observed.append(
            canonical_hash({
                "ids": sorted(
                    entry.scenario.scenario_id for entry in library
                ),
                "digests": sorted(
                    entry.digest for entry in library
                ),
            })
        )

    assert len(set(observed)) == 1

check("50 repeated library loads are deterministic", test_repeated_determinism)

# ---------------------------------------------------------------------
# 13. Clean wheel build without installation into Colab.
# ---------------------------------------------------------------------
def test_wheel_build():
    wheel_dir = Path("/tmp/dccp-direct-wheel")
    if wheel_dir.exists():
        shutil.rmtree(wheel_dir)
    wheel_dir.mkdir(parents=True)

    p = subprocess.run(
        [
            sys.executable, "-m", "pip", "wheel",
            "--no-deps",
            str(ROOT),
            "-w", str(wheel_dir),
        ],
        cwd="/content",
        env=TEST_ENV,
        text=True,
        capture_output=True,
        timeout=300,
    )

    assert p.returncode == 0, (p.stdout + "\n" + p.stderr)[-12000:]
    wheels = list(wheel_dir.glob("*.whl"))
    assert len(wheels) == 1, wheels

check("package wheel builds successfully", test_wheel_build)

# ---------------------------------------------------------------------
# 14. Final report.
# ---------------------------------------------------------------------
report = {
    "tested_at_utc": datetime.now(timezone.utc).isoformat(),
    "commit": COMMIT,
    "total_tests": len(results),
    "passed": sum(item["passed"] for item in results),
    "failed": sum(not item["passed"] for item in results),
    "results": results,
}

report_path = ROOT / "colab_runtime_results.json"
report_path.write_text(
    json.dumps(report, indent=2) + "\n",
    encoding="utf-8",
)

print()
print("=" * 90)
print("FINAL RESULT")
print("=" * 90)
print(json.dumps(
    {
        "commit": report["commit"],
        "total_tests": report["total_tests"],
        "passed": report["passed"],
        "failed": report["failed"],
    },
    indent=2,
))

if report["failed"]:
    print()
    print("=" * 90)
    print("FAILED TEST DETAILS")
    print("=" * 90)
    for item in report["results"]:
        if not item["passed"]:
            print("\nTEST:", item["test"])
            print(item["detail"])
